In [1]:
# imports and setup

import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time
from tqdm import tqdm
import mariadb

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

In [2]:

class AppConstants:
    """Constantes globais para a aplicação."""

    BEDROCK_DEFAULT_MODEL_ID = "anthropic.claude-3-7-sonnet-20240729-v1:0"
    # DEFAULT_PROMPT_EXTRACAO_PATH = "prompt_extracao_laudo.txt"
    # DEFAULT_PROMPT_RESUMO_PATH = "prompt_resumo.txt"
    # S3_BUCKET_NAME = "agente-ai-laudos"
    # S3_RESULTS_PREFIX = "resultados"
    # S3_DEBUG_PREFIX = "debug"
    MAX_RETRIES = 5
    INITIAL_BACKOFF_SECONDS = 2


In [3]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"


In [4]:
def criar_boto3_client(
    service_name: str, settings: Settings, config: Optional[Config] = None
) -> boto3.client:
    try:
        logger.info(
            f"Criando cliente {service_name.upper()} para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        client = boto3.client(
            service_name,
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
            config=config,
        )
        logger.info(f"Cliente {service_name.upper()} criado com sucesso.")
        return client
    except Exception as e:
        logger.critical(f"Não foi possível criar o cliente {service_name.upper()}: {e}")
        raise


In [5]:
def load_query_from_file(file_path: str) -> str:
    """
    Load SQL query from a text file
    
    Args:
        file_path: Path to the query file
    
    Returns:
        Query string
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            
        # Extract just the query part (remove variable assignment)
        if '"""' in content:
            # Find the query between triple quotes
            start = content.find('"""') + 3
            end = content.rfind('"""')
            query = content[start:end].strip()
        else:
            # If no triple quotes, assume entire file is the query
            query = content.strip()
            
        return query
        
    except Exception as e:
        logger.error(f"❌ Error loading query from file: {e}")
        return None

In [6]:
# connection to the database

class OracleService:
    def __init__(self, settings: Settings):
        self.config = {
            "user": settings.ORACLE_USER,
            "password": settings.ORACLE_PASSWORD,
            "dsn": settings.ORACLE_DSN,
        }
        self.connection = None
        if settings.ORACLE_INSTANT_CLIENT_PATH and os.path.isdir(
            settings.ORACLE_INSTANT_CLIENT_PATH
        ):
            logger.info(
                f"Inicializando Oracle Client de: {settings.ORACLE_INSTANT_CLIENT_PATH}"
            )
            oracledb.init_oracle_client(lib_dir=settings.ORACLE_INSTANT_CLIENT_PATH)

    def __enter__(self):
        """Establishes the database connection when entering the 'with' block."""
        try:
            self.connection = oracledb.connect(**self.config)
            logger.info(f"✅ Connection established to {self.config['dsn']}")
            return self
        except Exception as e:
            logger.error(f"❌ Error connecting to Oracle: {e}")
            raise

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Closes the database connection when exiting the 'with' block."""
        if self.connection:
            try:
                self.connection.close()
                logger.info("✅ Oracle connection closed.")
            except Exception as e:
                logger.error(f"❌ Error closing Oracle connection: {e}")
        # If an exception occurred within the 'with' block, it will be re-raised.

    def execute_query(self, query: str, fetch_limit: int = None) -> Optional[List[Dict[str, Any]]]:
        """
        Execute a query using the existing Oracle connection.
        
        Args:
            query: SQL query to execute
            fetch_limit: Maximum number of rows to fetch (None for all)
        
        Returns:
            List of dictionaries containing query results, or None if error
        """
        if not self.connection:
            logger.error("❌ Cannot execute query: Not connected. Use within a 'with' block.")
            return None
            
        try:
            with self.connection.cursor() as cursor:
                logger.info("Executing query...")
                cursor.execute(query)
                
                columns = [desc[0] for desc in cursor.description]
                
                if fetch_limit:
                    rows = cursor.fetchmany(fetch_limit)
                else:
                    rows = cursor.fetchall()
                
                results = []
                for row in rows:
                    row_dict = {}
                    for i, col_val in enumerate(row):
                        col_name = columns[i]
                        if isinstance(col_val, oracledb.LOB):
                            row_dict[col_name] = col_val.read()
                        else:
                            row_dict[col_name] = col_val
                    results.append(row_dict)
                
                logger.info(f"✅ Fetched {len(results)} rows.")
                return results
                    
        except Exception as e:
            logger.error(f"Error executing query: {e}")
            return None

In [7]:
class MariaDBService:
    def __init__(self, settings: Settings):
        self.config = {
            "user": settings.MARIADB_USER,
            "password": settings.MARIADB_PASSWORD,
            "host": settings.MARIADB_HOST,
            "port": settings.MARIADB_PORT,
            "database": settings.MARIADB_DATABASE,
        }

    def buscar_avisos(self) -> List[Dict]:
        logger.info("Buscando lista de avisos de cirurgia do MariaDB...")
        query = """
        WITH db as ( SELECT distinct so.surgical_order_id, so.hospital_id, so.hospitalization_type, CASE so.patient_hospitalized WHEN 0 THEN 'Não' WHEN 1 THEN 'Sim' ELSE NULL END AS patient_hospitalized, h.friendly_name, h.friendly_name as filter_name_hospital, IFNULL(IFNULL(u.name, so.doctor_name), 'SEM REGISTRO') as doctor_name, d.specialty, so.patient_name, case when so.opme = '{"solicitations":[],"providers":[]}' then 'Sem OPME' when so.opme is null then 'Sem OPME' else 'Com OPME' end as with_opme, so.opme, case when CHAR_LENGTH(SUBSTRING_INDEX(SUBSTRING(so.procedure, 34, 100), '"', 1)) = 0 then SUBSTRING_INDEX(SUBSTRING(so.procedure, 78, 100), '"', 1) when CHAR_LENGTH(SUBSTRING_INDEX(SUBSTRING(so.procedure, 34, 100), '"', 1)) = 1 then SUBSTRING_INDEX(SUBSTRING(so.procedure, 36, 100), '"', 1) else SUBSTRING_INDEX(SUBSTRING(so.procedure, 34, 100), '"', 1) end as procedimento_teste, so.created_at, so.expected_date, ss.status as status, ss.provider, ss.created_at as dt_status, str_to_date(DATE_FORMAT(ss.created_at, '%d/%m/%Y'), '%d/%m/%Y') as dt_status_format, str_to_date(DATE_FORMAT(CONCAT(DATE_FORMAT(so.created_at, '%Y/%m/'), '01'), '%d/%m/%Y'), '%d/%m/%Y') as mes_ano_criacao, hi.health_insurance_code, hi.health_insurance_name, case when so.hospital_id in (1, 6, 7, 8, 12) then 'CLUSTER RMBH/SALVADOR' ELSE 'HUB UNIDADES' END AS cluster_hub from surgery_flow_prod.surgical_order so join surgery_flow_prod.hospital h on so.hospital_id = h.hospital_id left join surgery_flow_prod.doctor d on d.doctor_id = so.doctor_id left join surgery_flow_prod.user u on d.user_id = u.user_id LEFT join surgery_flow_prod.health_insurance hi on hi.health_insurance_id = so.health_insurance_id join surgery_flow_prod.surgical_status ss on ss.surgical_order_id = so.surgical_order_id and ss.is_active = true AND ss.status = 'Revisão' where 1 = 1 and so.is_complete = TRUE and so.deleted_at IS NULL and so.created_at >= DATE_SUB(CURDATE(), INTERVAL 2 MONTH) and so.hospitalization_mode not in ('ambulatory', 'emergency room') and so.patient_name not like 'RN %' and so.patient_hospitalized = false ) SELECT * FROM db where db.with_opme = 'Sem OPME' and db.health_insurance_code IN ('89', '20', '46', '110', '383', '391', '372')
        """
        try:
            with (
                mariadb.connect(**self.config) as conn,
                conn.cursor(dictionary=True) as cursor,
            ):
                cursor.execute(query)
                return cursor.fetchall()
        except mariadb.Error as e:
            logger.error(f"Erro ao buscar avisos no MariaDB: {e}")
            raise


In [8]:
class MariaDBService:
    """Context manager for MariaDB database connections."""
    
    def __init__(self, settings: Settings):
        self.config = {
            "user": settings.MARIADB_USER,
            "password": settings.MARIADB_PASSWORD,
            "host": settings.MARIADB_HOST,
            "port": settings.MARIADB_PORT,
            "database": settings.MARIADB_DATABASE,
        }
        self.connection = None

    def __enter__(self):
        """Establishes the database connection when entering the 'with' block."""
        try:
            self.connection = mariadb.connect(**self.config)
            logger.info(f"✅ MariaDB connection established to {self.config['host']}:{self.config['port']}/{self.config['database']}")
            return self
        except mariadb.Error as e:
            logger.error(f"❌ Error connecting to MariaDB: {e}")
            raise

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Closes the database connection when exiting the 'with' block."""
        if self.connection:
            try:
                self.connection.close()
                logger.info("✅ MariaDB connection closed.")
            except mariadb.Error as e:
                logger.error(f"❌ Error closing MariaDB connection: {e}")
        # If an exception occurred within the 'with' block, it will be re-raised.

    def execute_query(self, query: str, fetch_limit: int = None) -> Optional[List[Dict[str, Any]]]:
        """
        Execute a query using the existing MariaDB connection.
        
        Args:
            query: SQL query to execute
            fetch_limit: Maximum number of rows to fetch (None for all)
        
        Returns:
            List of dictionaries containing query results, or None if error
        """
        if not self.connection:
            logger.error("❌ Cannot execute query: Not connected. Use within a 'with' block.")
            return None
            
        try:
            with self.connection.cursor(dictionary=True) as cursor:
                logger.info("Executing MariaDB query...")
                cursor.execute(query)
                
                if fetch_limit:
                    results = cursor.fetchmany(fetch_limit)
                else:
                    results = cursor.fetchall()
                
                logger.info(f"✅ Fetched {len(results)} rows from MariaDB.")
                return results
                    
        except mariadb.Error as e:
            logger.error(f"Error executing MariaDB query: {e}")
            return None

    

In [9]:
# Load the carteirinha query from file
query_file_path = "/home/joao/projects/company_projects/carteirinha-api/documents/querys/query_carteirinha.txt"
convenio_carteirinha_query = load_query_from_file(query_file_path)

if convenio_carteirinha_query:
    logger.info("✅ Carteirinha query loaded successfully from file!")
else:
    logger.error("❌ Failed to load query from file")

{"timestamp": "2025-08-07T10:24:59", "level": "INFO", "name": "__main__", "message": "✅ Carteirinha query loaded successfully from file!", "filename": "2412977378.py", "lineno": 6}


In [10]:
# aviso cirurgia query file
aviso_cirurgia_query_file = "/home/joao/projects/company_projects/carteirinha-api/documents/querys/query_aviso_cirurgia.txt"
if os.path.exists(aviso_cirurgia_query_file):
    aviso_cirurgia_query = load_query_from_file(aviso_cirurgia_query_file)
    if aviso_cirurgia_query:
        logger.info("✅ Aviso cirurgia query loaded successfully from file!")
    else:
        logger.error("❌ Failed to load aviso cirurgia query from file")
else:
    logger.error(f"❌ Aviso cirurgia query file not found: {aviso_cirurgia_query_file}")

{"timestamp": "2025-08-07T10:24:59", "level": "INFO", "name": "__main__", "message": "✅ Aviso cirurgia query loaded successfully from file!", "filename": "3187017494.py", "lineno": 6}


In [11]:
convenio_carteirinha_query

"SELECT\n    dac.cd_aviso_cirurgia,\n    dac.CD_DOCUMENTO_ANEXO_CIRURGICO,\n    g.cd_guia,\n    g.tp_guia,\n    g.tp_situacao,\n    dac.LO_DOCUMENTO_ANEXO_CIRURGICO,\n    dac.DS_EXTENSAO,\n    dac.DT_ANEXO,\n    dac.ds_documento_anexo\nFROM\n    dbamv.DOCUMENTO_ANEXO_CIRURGICO dac\nJOIN\n    dbamv.guia g\nON\n    dac.cd_aviso_cirurgia = g.cd_aviso_cirurgia\nWHERE\n    dac.cd_usuario = 'MMD_INTEGRACAO'\n    AND dac.DS_DOCUMENTO_ANEXO LIKE '%Carteira do convênio'\nAND\n    dac.cd_aviso_cirurgia\nIN\n(\n    836905, 842575, 844359, 844391, 845763, 845857, 846206, 847154, 847884, 848690,\n    849847, 849914, 850203, 851314, 851798, 851807, 851816, 851990, 852029, 852049,\n    852297, 852300, 852544, 852721, 852796, 852965, 853039, 853116, 853136, 853282,\n    853333, 853967, 854136, 854158, 854434, 854777, 855272, 855319, 855334, 855526,\n    855640, 856244, 856478, 856640, 856669, 856714, 856797, 856820, 856920, 856929,\n    856954, 857003, 857008, 857128, 857187, 857233, 857282, 857387, 8

In [12]:
#  Carteirinha Query using the OracleService context manager

logger.info("🚀 Testing carteirinha query with context manager...")
logger.info("=" * 50)

results = None
try:
    settings = Settings()
    
    # Use the service as a context manager
    with OracleService(settings) as oracle_service:
        if convenio_carteirinha_query:
            results = oracle_service.execute_query(convenio_carteirinha_query, fetch_limit=5)

            if results:
                logger.info(f"\n✅ Query executed successfully! Found {len(results)} sample records")
                logger.info("\n📋 Sample Results:")
                logger.info("-" * 50)
                
                for i, record in enumerate(results, 1):
                    logger.info(f"\nRecord {i}:")
                    for key, value in record.items():
                        if key == 'LO_DOCUMENTO_ANEXO_CIRURGICO':
                            logger.info(f"  {key}: {'BLOB data present' if value else 'No BLOB data'}")
                        else:
                            logger.info(f"  {key}: {value}")
                
                logger.info(f"\n📊 To get all records, run the query without fetch_limit.")
                
            else:
                logger.error("❌ No results returned or query failed inside 'with' block")
        else:
            logger.error("❌ Query not loaded - cannot execute")

except Exception as e:
    logger.error(f"❌ An error occurred outside the 'with' block: {e}")

logger.info("=" * 50)

{"timestamp": "2025-08-07T10:24:59", "level": "INFO", "name": "__main__", "message": "🚀 Testing carteirinha query with context manager...", "filename": "754050125.py", "lineno": 3}
{"timestamp": "2025-08-07T10:24:59", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "754050125.py", "lineno": 4}
{"timestamp": "2025-08-07T10:25:00", "level": "INFO", "name": "__main__", "message": "✅ Connection established to srvhmddb007-otk6s-scan.sbntdb.vcnprod.oraclevcn.com:1521/PRDREPT_OCI", "filename": "1417701281.py", "lineno": 23}
{"timestamp": "2025-08-07T10:25:00", "level": "INFO", "name": "__main__", "message": "Executing query...", "filename": "1417701281.py", "lineno": 56}
{"timestamp": "2025-08-07T10:25:09", "level": "INFO", "name": "__main__", "message": "✅ Fetched 5 rows.", "filename": "1417701281.py", "lineno": 77}
{"timestamp": "2025-08-07T10:25:09", "level": "INFO", "name": "__main__", "message": "\n✅ Query executed success

In [13]:
# avisos query using maria db context manager
logger.info("🚀 Testing avisos query with MariaDB context manager...")
logger.info("=" * 50)
try:
    settings = Settings()
    
    # Use the service as a context manager
    with MariaDBService(settings) as maria_service:
        maria_results = maria_service.execute_query(aviso_cirurgia_query, fetch_limit=5)
        if maria_results:
            logger.info(f"\n✅ Avisos query executed successfully! Found {len(maria_results)} sample records")
            logger.info("\n📋 Sample Results:")
            logger.info("-" * 50)
            
            for i, record in enumerate(maria_results, 1):
                logger.info(f"\nRecord {i}:")
                for key, value in record.items():
                    logger.info(f"  {key}: {value}")
                    
            logger.info(f"\n📊 To get all records, run the query without fetch_limit.")
            
        else:
            logger.error("❌ No results returned or query failed inside 'with' block")
except Exception as e:
    logger.error(f"❌ An error occurred while executing avisos query: {e}")



{"timestamp": "2025-08-07T10:25:09", "level": "INFO", "name": "__main__", "message": "🚀 Testing avisos query with MariaDB context manager...", "filename": "2891619402.py", "lineno": 2}
{"timestamp": "2025-08-07T10:25:09", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "2891619402.py", "lineno": 3}
{"timestamp": "2025-08-07T10:25:10", "level": "INFO", "name": "__main__", "message": "✅ MariaDB connection established to srvawsdb002.cow7tj30bxpl.us-east-1.rds.amazonaws.com:3306/", "filename": "2645936225.py", "lineno": 18}
{"timestamp": "2025-08-07T10:25:10", "level": "INFO", "name": "__main__", "message": "Executing MariaDB query...", "filename": "2645936225.py", "lineno": 51}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Fetched 5 rows from MariaDB.", "filename": "2645936225.py", "lineno": 59}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "\n

In [14]:
# image utils
def converter_blob_para_imagens(blob: bytes, extensao: str) -> List[Image.Image]:
    imagens = []
    ext = extensao.lower().strip(".") if extensao else ""
    try:
        if ext == "pdf":
            with fitz.open(stream=blob, filetype="pdf") as pdf_doc:
                logger.info(f"Processando PDF com {len(pdf_doc)} página(s)...")
                for pagina in pdf_doc:
                    pix = pagina.get_pixmap(matrix=fitz.Matrix(3.0, 3.0), alpha=False)
                    imagens.append(Image.open(io.BytesIO(pix.tobytes("png"))))
        elif ext in ["jpg", "jpeg", "png", "bmp"]:
            imagens.append(Image.open(io.BytesIO(blob)))
        else:
            logger.warning(f"Formato de arquivo não suportado: '{ext}'.")
    except Exception as e:
        logger.error(f"Erro ao converter BLOB para imagem (ext: .{ext}): {e}")
    return imagens


def aplicar_clahe(imagem: Image.Image) -> Image.Image:
    try:
        imagem_cv = cv2.cvtColor(np.array(imagem), cv2.COLOR_RGB2BGR)
        imagem_cinza = cv2.cvtColor(imagem_cv, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return Image.fromarray(clahe.apply(imagem_cinza))
    except Exception:
        return imagem


def imagem_para_base64(imagem: Image.Image) -> str:
    TARGET_BYTES = 3.5 * 1024 * 1024
    if imagem.mode in ("RGBA", "P"):
        imagem = imagem.convert("RGB")
    for quality in range(95, 15, -10):
        buffer = io.BytesIO()
        imagem.save(buffer, format="JPEG", quality=quality)
        if buffer.tell() <= TARGET_BYTES:
            if quality < 95:
                logger.warning(
                    f"Imagem comprimida (qualidade {quality}%) para caber no limite."
                )
            return base64.b64encode(buffer.getvalue()).decode("utf-8")
    raise ValueError(
        f"Não foi possível reduzir a imagem abaixo de {TARGET_BYTES / (1024 * 1024):.1f}MB."
    )


In [15]:
# extract blob data from the results
blobs = []
if results:
    for record in results:
        blob_data = record.get('LO_DOCUMENTO_ANEXO_CIRURGICO')
        if blob_data and isinstance(blob_data, bytes):
            blobs.append(blob_data)
            logger.info(f"✅ Extracted BLOB of size {len(blob_data)} bytes")
        else:
            logger.warning("No BLOB data found in record or data is not bytes")

if blobs:
    logger.info(f"Successfully extracted {len(blobs)} BLOBs into a list.")
else:
    logger.error("Could not extract any BLOBs from the results.")

{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 506822 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 4224600 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 261028 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 6864055 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 216457 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "Successfully extracted 5 BLOBs into a list.", "filename": "395405530.py", "li

In [16]:
# take the list of blobs and convert them to images. them, save them as base64 strings
base64_images = []
if blobs:
    for i, blob in enumerate(blobs):
        logger.info(f"Converting BLOB {i+1}/{len(blobs)} to images...")
        imagens = converter_blob_para_imagens(blob, "pdf")  # Assuming PDF for this example
        # save the images on ../documents/carteirinhas_images/
        if not os.path.exists("../documents/carteirinhas_images/"):
            os.makedirs("../documents/carteirinhas_images/")
        for j, img in enumerate(imagens):
            img_path = f"../documents/carteirinhas_images/blob_{i+1}_image_{j+1}.png"
            img.save(img_path)
            logger.info(f"Saved image {j+1} from BLOB {i+1} to {img_path}")
        logger.info(f"Extracted {len(imagens)} images from BLOB {i+1}.")

        if imagens:
            for j, img in enumerate(imagens):
                logger.info(f"Processing image {j+1} from BLOB {i+1}...")
                img_clahe = aplicar_clahe(img)
                base64_str = imagem_para_base64(img_clahe)
                base64_images.append(base64_str)
                logger.info(f"✅ Converted image {j+1} to base64 string.")
        else:
            logger.warning(f"No images extracted from BLOB {i+1}.")

{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "Converting BLOB 1/5 to images...", "filename": "4197694994.py", "lineno": 5}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "Processando PDF com 1 página(s)...", "filename": "2299205648.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:12", "level": "INFO", "name": "__main__", "message": "Saved image 1 from BLOB 1 to ../documents/carteirinhas_images/blob_1_image_1.png", "filename": "4197694994.py", "lineno": 13}
{"timestamp": "2025-08-07T10:25:12", "level": "INFO", "name": "__main__", "message": "Extracted 1 images from BLOB 1.", "filename": "4197694994.py", "lineno": 14}
{"timestamp": "2025-08-07T10:25:12", "level": "INFO", "name": "__main__", "message": "Processing image 1 from BLOB 1...", "filename": "4197694994.py", "lineno": 18}
{"timestamp": "2025-08-07T10:25:12", "level": "INFO", "name": "__main__", "message": "✅ Converted image 1 to base64 string.", "filen

In [17]:
base64_images_count = len(base64_images)
if base64_images_count > 0:
    logger.info(f"Successfully converted {base64_images_count} images to base64 strings.")
else:
    logger.warning("No images were converted to base64 strings. Check the BLOB data.")

{"timestamp": "2025-08-07T10:25:16", "level": "INFO", "name": "__main__", "message": "Successfully converted 5 images to base64 strings.", "filename": "1948280603.py", "lineno": 3}


In [18]:
class CarteirinhaExtraida(PydanticBaseModel):
    """Define a estrutura dos dados extraídos da carteirinha."""
    convenio: Optional[str] = Field(None, description="Nome do convênio de saúde.")
    plano: Optional[str] = Field(None, description="Nome do plano de saúde.")
    nome_pessoa: Optional[str] = Field(None, description="Nome completo do titular ou beneficiário.")
    numero_carteirinha: Optional[str] = Field(None, description="O número de identificação da carteirinha.")

In [19]:
class LLMService:
    """Encapsula a lógica de chamada ao modelo de linguagem (Bedrock)."""

    def __init__(self, bedrock_client: boto3.client, model_id: str, data_model: PydanticBaseModel):
        self.bedrock_client = bedrock_client
        self.model_id = model_id
        self.data_model = data_model

    def _extract_json_from_response(self, raw_text: str) -> str:
        """
        Extrai JSON de diferentes formatos de resposta do LLM.
        Tenta múltiplos métodos de extração para maximizar compatibilidade.
        """
        # Method 1: JSON em blocos de código (```json ... ```)
        code_block_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_text, re.DOTALL)
        if code_block_match:
            logger.info("JSON extraído de bloco de código")
            return code_block_match.group(1).strip()
        
        # Method 2: JSON standalone (sem blocos de código)
        json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if json_match:
            logger.info("JSON extraído diretamente do texto")
            return json_match.group(0).strip()
        
        # # Method 3: Busca mais específica para estrutura esperada
        # specific_match = re.search(
        #     r'\{\s*"convenio".*?"numero_carteirinha".*?\}', 
        #     raw_text, 
        #     re.DOTALL | re.IGNORECASE
        # )
        # if specific_match:
        #     logger.info("JSON extraído por busca específica")
        #     return specific_match.group(0).strip()
        
        raise ValueError("Nenhum JSON válido encontrado na resposta do LLM")

    def _validate_and_clean_json(self, json_str: str) -> str:
        """
        Valida e limpa a string JSON antes da validação Pydantic.
        """
        try:
            # Tenta fazer parse para verificar se é JSON válido
            parsed = json.loads(json_str)
            # Se chegou até aqui, o JSON é válido
            return json_str
        except json.JSONDecodeError as e:
            logger.warning(f"JSON inválido detectado: {e}")
            # Tenta algumas correções comuns
            cleaned = json_str.strip()
            # Remove possíveis caracteres extras no início/fim
            cleaned = re.sub(r'^[^\{]*', '', cleaned)
            cleaned = re.sub(r'[^\}]*$', '', cleaned)
            try:
                json.loads(cleaned)
                return cleaned
            except json.JSONDecodeError:
                raise ValueError(f"Não foi possível corrigir o JSON: {e}")

    def _invocar_llm(self, corpo_requisicao: Dict, context_log: str = "") -> Dict:
        resultado = {
            "success": False,
            "dados": None,
            "raw_response": "",
            "error": None,
            "input_tokens": 0,
            "output_tokens": 0,
        }
        retries = 0
        while retries < AppConstants.MAX_RETRIES:
            try:
                response = self.bedrock_client.invoke_model(
                    body=json.dumps(corpo_requisicao), modelId=self.model_id
                )
                response_body = json.loads(response.get("body").read())
                usage = response_body.get("usage", {})
                raw_text = response_body.get("content", [{}])[0].get("text", "")
                
                resultado.update({
                    "input_tokens": usage.get("input_tokens", 0),
                    "output_tokens": usage.get("output_tokens", 0),
                    "raw_response": raw_text,
                })
                
                # Extração e validação do JSON
                json_str = self._extract_json_from_response(raw_text)
                cleaned_json = self._validate_and_clean_json(json_str)
                
                resultado["dados"] = cleaned_json
                resultado["success"] = True
                return resultado

            except self.bedrock_client.exceptions.ThrottlingException as e:
                retries += 1
                wait_time = AppConstants.INITIAL_BACKOFF_SECONDS * (2 ** (retries - 1))
                logger.warning(
                    f"LLM Throttling para {context_log}. Tentativa {retries}/{AppConstants.MAX_RETRIES}. "
                    f"Aguardando {wait_time:.2f}s. Erro: {e}"
                )
                time.sleep(wait_time)
            except Exception as e:
                resultado["error"] = str(e)
                logger.error(f"Erro na invocação do LLM para {context_log}: {resultado['error']}")
                return resultado
                
        resultado["error"] = f"Falha no LLM após {AppConstants.MAX_RETRIES} tentativas."
        logger.error(resultado["error"])
        return resultado

    def extrair_dados_de_imagem(self, prompt: str, imagem_base64: str, context_log: str) -> Dict:
        """
        Envia uma imagem e um prompt para o LLM e valida a resposta com o modelo Pydantic.
        """
        corpo = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 4096,
            "temperature": 0.05,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/jpeg",
                                "data": imagem_base64,
                            },
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
        }
        
        resultado = self._invocar_llm(corpo, context_log)
        
        if resultado["success"]:
            try:
                llm_output_json = resultado["dados"]  # JSON string limpo
                
                # Debug: Log do que estamos tentando validar
                logger.debug(f"Raw LLM response for {context_log}: {resultado['raw_response'][:200]}...")
                logger.debug(f"Extracted JSON for {context_log}: {llm_output_json}")
                
                # Validação Pydantic
                parsed_output = self.data_model.model_validate_json(llm_output_json)
                resultado["dados"] = parsed_output
                
                # Log dos dados extraídos para debug
                logger.info(f"✅ Dados extraídos para {context_log}:")
                logger.info(f"  Convênio: {parsed_output.convenio}")
                logger.info(f"  Plano: {parsed_output.plano}")
                logger.info(f"  Nome: {parsed_output.nome_pessoa}")
                logger.info(f"  Número: {parsed_output.numero_carteirinha}")
                
            except Exception as pydantic_error:
                logger.error(f"❌ Erro de validação Pydantic para {context_log}: {pydantic_error}")
                logger.error(f"JSON problemático: {llm_output_json}")
                logger.error(f"Raw response: {resultado['raw_response']}")
                resultado.update({
                    "success": False,
                    "error": f"Erro de validação Pydantic: {pydantic_error}",
                })
        
        return resultado

In [20]:
class ExtractionExperiment:
    """
    Encapsula a lógica para executar um experimento de extração de dados
    usando um modelo e prompt específicos.
    """
    def __init__(self, settings: Settings, model_id: str, prompt: str, prompt_name: str, data_model: PydanticBaseModel):
        self.settings = settings
        self.model_id = model_id
        self.prompt = prompt
        self.prompt_name = prompt_name
        self.data_model = data_model
        self.llm_service = self._initialize_llm_service()
        self.results = []

    def _initialize_llm_service(self) -> Optional[LLMService]:
        """Inicializa o serviço LLM para o modelo especificado."""
        try:
            bedrock_client = criar_boto3_client("bedrock-runtime", self.settings)
            llm_service = LLMService(bedrock_client=bedrock_client, model_id=self.model_id, data_model=self.data_model)
            logger.info(f"✅ Serviço LLM inicializado para o modelo: {self.model_id}")
            return llm_service
        except Exception as e:
            logger.critical(f"❌ Falha ao inicializar o serviço LLM para {self.model_id}: {e}")
            return None

    def run(self, images_base64: List[str]):
        """Executa o experimento de extração nas imagens fornecidas."""
        if not self.llm_service:
            logger.error("Não é possível executar o experimento, o serviço LLM não foi inicializado.")
            return

        logger.info(f"🚀 Executando experimento com o modelo '{self.model_id}' e prompt '{self.prompt_name}'...")

        for i, b64_image in tqdm(enumerate(images_base64)):
            context_log = f"imagem_{i+1}"
            logger.info(f"--- Processando {context_log} ---")
            
            extraction_result = self.llm_service.extrair_dados_de_imagem(
                prompt=self.prompt,
                imagem_base64=b64_image,
                context_log=context_log
            )
            
            if extraction_result["success"]:
                dados = extraction_result["dados"]
                logger.info("✅ Extração bem-sucedida!")
                self.results.append(dados.model_dump())
            else:
                logger.error(f"❌ Falha na extração para {context_log}: {extraction_result['error']}")
        
        logger.info(f"✅ Experimento finalizado. {len(self.results)} extrações bem-sucedidas.")
        self.save_results()

    def save_results(self):
        """Salva os resultados da extração em um arquivo JSON com nome dinâmico."""
        if not self.results:
            logger.warning("⚠️ Nenhum resultado para salvar.")
            return

        model_name_safe = self.model_id.replace(":", "_").replace(".", "_")
        prompt_name_safe = self.prompt_name.replace(" ", "_").lower()
        
        filename = f"extract_results_{model_name_safe}_{prompt_name_safe}.json"
        results_dir = "../documents/extraction_results"
        os.makedirs(results_dir, exist_ok=True)
        results_path = os.path.join(results_dir, filename)
        
        try:
            with open(results_path, 'w', encoding='utf-8') as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)
            logger.info(f"✅ Resultados salvos em: {results_path}")
        except Exception as e:
            logger.error(f"❌ Falha ao salvar o arquivo JSON: {e}")

In [21]:
multimodal_llms = ["us.anthropic.claude-3-7-sonnet-20250219-v1:0"] # other models we need access.

In [22]:
prompts = {
    "prompt_extracao_carteirinha_v1": """
        A imagem fornecida é uma carteirinha de convênio de saúde. Analise a imagem e extraia as seguintes informações em formato JSON:
        - convenio: O nome da operadora do plano de saúde.
        - plano: O tipo ou nome do plano (ex: "Plano Prata", "Enfermaria").
        - nome_pessoa: O nome completo do beneficiário.
        - numero_carteirinha: O número de identificação ou matrícula da carteirinha.

        Se alguma informação não for encontrada, retorne `null` para o campo correspondente.
        O JSON deve ter a seguinte estrutura:
        {
        "convenio": "string",
        "plano": "string",
        "nome_pessoa": "string",
        "numero_carteirinha": "string"
        }
        """
}

In [23]:
# Quick Model Availability Test

def test_model_availability(model_ids: List[str], settings: Settings) -> List[str]:
    """Test which models are available and return only the working ones."""
    available_models = []
    
    try:
        bedrock_client = criar_boto3_client("bedrock-runtime", settings)
        
        for model_id in model_ids:
            logger.info(f"🧪 Testing model: {model_id}")
            
            # Simple test payload
            test_payload = {
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 10,
                "temperature": 0.1,
                "messages": [
                    {
                        "role": "user",
                        "content": [{"type": "text", "text": "Hello"}]
                    }
                ]
            }
            
            try:
                response = bedrock_client.invoke_model(
                    body=json.dumps(test_payload), 
                    modelId=model_id
                )
                logger.info(f"✅ Model {model_id} is AVAILABLE")
                available_models.append(model_id)
                
            except bedrock_client.exceptions.ValidationException as e:
                if "on-demand throughput" in str(e):
                    logger.warning(f"❌ Model {model_id} requires special access/provisioning")
                else:
                    logger.warning(f"❌ Model {model_id} validation error: {e}")
            except Exception as e:
                logger.error(f"❌ Model {model_id} failed: {e}")
                
    except Exception as e:
        logger.error(f"❌ Error testing models: {e}")
    
    return available_models

# Test your models
logger.info("🚀 Testing model availability...")
logger.info("=" * 50)

settings = Settings()
test_models = ["us.anthropic.claude-3-7-sonnet-20250219-v1:0", "anthropic.claude-sonnet-4-20250514-v1:0"]

available_models = test_model_availability(test_models, settings)

if available_models:
    logger.info(f"✅ Available models: {available_models}")
    # Update your multimodal_llms list with only working models
    multimodal_llms = available_models
else:
    logger.error("❌ No models available!")
    
logger.info("=" * 50)

{"timestamp": "2025-08-07T10:25:16", "level": "INFO", "name": "__main__", "message": "🚀 Testing model availability...", "filename": "1572756103.py", "lineno": 48}
{"timestamp": "2025-08-07T10:25:16", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "1572756103.py", "lineno": 49}
{"timestamp": "2025-08-07T10:25:16", "level": "INFO", "name": "__main__", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-07T10:25:16", "level": "INFO", "name": "__main__", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "788707453.py", "lineno": 15}
{"timestamp": "2025-08-07T10:25:16", "level": "INFO", "name": "__main__", "message": "🧪 Testing model: us.anthropic.claude-3-7-sonnet-20250219-v1:0", "filename": "1572756103.py", "lineno": 11}
{"timestamp": "2025-08-07T10:25:17", "level": "INFO", "name": "__main__", "message": "✅ Model us.an

In [24]:
# Executar o Experimento de Extração

logger.info("🚀 Configurando e executando o experimento de extração...")
logger.info("=" * 50)

# 1. Definição do Prompt e Nome do Prompt
extraction_prompt = prompts["prompt_extracao_carteirinha_v1"]
prompt_name = "prompt_extracao_carteirinha_v1"


# 2. Verificação dos pré-requisitos
if 'base64_images' in locals() and base64_images:
    try:
        for model_id in multimodal_llms:
            logger.info(f"Testando o modelo: {model_id}")
            # 3. Inicialização e Execução do Experimento
            settings = Settings()
            
            # Use o modelo padrão das configurações
            model_to_test = model_id
            
            experiment = ExtractionExperiment(
                settings=settings,
                model_id=model_to_test,
                prompt=extraction_prompt,
                prompt_name=prompt_name,
                data_model=CarteirinhaExtraida
            )
            
            experiment.run(base64_images)

    except Exception as e:
        logger.critical(f"❌ Ocorreu um erro crítico durante a configuração do experimento: {e}")
else:
    logger.warning("⚠️ Nenhuma imagem em base64 foi encontrada para processar. Execute as células anteriores para gerar a variável 'base64_images'.")

logger.info("=" * 50)
logger.info("✅ Processo de experimento finalizado.")

{"timestamp": "2025-08-07T10:25:18", "level": "INFO", "name": "__main__", "message": "🚀 Configurando e executando o experimento de extração...", "filename": "3228186377.py", "lineno": 3}
{"timestamp": "2025-08-07T10:25:18", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "3228186377.py", "lineno": 4}
{"timestamp": "2025-08-07T10:25:18", "level": "INFO", "name": "__main__", "message": "Testando o modelo: us.anthropic.claude-3-7-sonnet-20250219-v1:0", "filename": "3228186377.py", "lineno": 15}
{"timestamp": "2025-08-07T10:25:18", "level": "INFO", "name": "__main__", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-07T10:25:18", "level": "INFO", "name": "__main__", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "788707453.py", "lineno": 15}
{"timestamp": "2025-08-07T10:25:18", "level": "INFO", "name": "__main__", 